# 02 — Data forensics

Duplicate payments, attribution, timezones, vendors, agents, mix, denominators.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
GOLDEN = ROOT / 'data' / 'golden'
CLEAN = ROOT / 'data' / 'clean'
plt.rcParams['figure.figsize'] = (9, 4)
assert (GOLDEN / 'metrics_monthly.parquet').exists(), 'Run python scripts/run_pipeline.py first'


In [ ]:

pay = pd.read_csv(RAW/'payments.csv')
print('raw rows', len(pay), 'unique payment_id', pay.payment_id.nunique(), 'exact dups', pay.duplicated().sum())
print('SUCCESS raw ₹', pay.loc[pay.payment_status=='SUCCESS','amount'].sum())
print('SUCCESS unique id ₹', pay.drop_duplicates('payment_id').query('payment_status=="SUCCESS"').amount.sum())
ref = pay.groupby('payment_reference')['account_id'].nunique()
print('references with >1 account', int((ref>1).sum()))


In [ ]:

calls = pd.read_csv(RAW/'calls.csv').drop_duplicates('call_id')
calls['event_at'] = pd.to_datetime(calls.event_at)
parts=[]
for tz, g in calls.groupby('timezone'):
    parts.append(g.event_at.dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT').dt.tz_convert('Asia/Kolkata'))
calls['ist'] = pd.concat(parts)
print('call date shift rate', (calls.event_at.dt.date != calls.ist.dt.date).mean())


In [ ]:

ag = pd.read_csv(RAW/'agents.csv')
print(ag.groupby('agent_id').nunique()[['employee_code','agent_name','vendor_id','team','status']].gt(1).mean())
print('distinct names', ag.agent_name.nunique())


Full treatment and severity: `docs/data_quality_report.md`.